# Análisis de customers.json

**Autor:** Daniel Guzmán  
**Fecha:** 2026-04-23  
**Entorno:** Databricks

In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, when

In [0]:
catalog = "workspace"
schema = "default"
volume = "customers_files_daniel"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

print(path_volume)

In [0]:
inicio = time.time()

df_json = (
    spark.read
    .option("multiline", True)
    .json(f"{path_volume}/customers.json")
)

total_registros = df_json.count()
fin = time.time()

print(f"Tiempo de lectura: {fin - inicio:.4f} segundos")
print(f"Registros: {total_registros}")
print(f"Columnas: {df_json.columns}")

In [0]:
display(df_json.limit(5))

In [0]:
print("Tipos de datos:")
for col_name, dtype in df_json.dtypes:
    print(f"{col_name}: {dtype}")

In [0]:
print("Shape:")
print(f"Filas: {df_json.count()}")
print(f"Columnas: {len(df_json.columns)}")

In [0]:
df_json.printSchema()

In [0]:
nulos_df = df_json.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_json.columns
])

display(nulos_df)

In [0]:
for c in df_json.columns:
    print(f"{c}: {df_json.select(c).distinct().count()} valores únicos")

In [0]:
display(df_json.describe())

In [0]:
clientes_por_pais = (
    df_json.groupBy("Country")
    .count()
    .withColumnRenamed("count", "total_clientes")
    .orderBy(F.col("total_clientes").desc())
)

display(clientes_por_pais)

In [0]:
display(clientes_por_pais.limit(5))

In [0]:
empresas_distintas = df_json.select("Company").distinct().count()
print(f"Empresas distintas: {empresas_distintas}")

In [0]:
nombres_frecuentes = (
    df_json.withColumn("Nombre Completo", F.concat_ws(" ", F.col("First Name"), F.col("Last Name")))
    .groupBy("Nombre Completo")
    .count()
    .orderBy(F.col("count").desc())
)

display(nombres_frecuentes.limit(1))

In [0]:
clientes_sin_ciudad = df_json.filter(
    F.col("City").isNull() | (F.trim(F.col("City")) == "")
).count()

print(f"Clientes sin ciudad registrada: {clientes_sin_ciudad}")

## Reflexión sobre el formato .json

- Tiempo de lectura registrado: 2.6327 segundos
- Tamaño del archivo: 4.47 MB aprox.
- Fue fácil de leer con PySpark en Databricks.
- Como ventaja, JSON es flexible y útil para datos semiestructurados. Como desventaja, suele ser más pesado y menos eficiente para análisis que formatos columnar como Parquet.
- Usaría JSON cuando necesite intercambiar datos con APIs, eventos o estructuras jerárquicas.